[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/bloc2_donnees/exercices/seance2_exercices.ipynb)

# Séance 2.2 — Agréger, croiser et visualiser

**Exercices** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- répondre à « combien par... ? » avec `groupby`, et calculer plusieurs indicateurs d'un coup
- rassembler plusieurs fichiers avec `merge`, sans perdre ni dupliquer de lignes
- choisir le bon graphique selon la question posée, et le rendre lisible
- repérer ce qu'un graphique cache autant que ce qu'il montre
- conclure une analyse par des recommandations chiffrées

## Comment ça marche

La feuille compte **deux parties**, à faire dans l'ordre.

**Partie 1 — l'échauffement.** Le code est déjà écrit, il ne reste que les `____` à
remplir. Chaque exercice se termine par une cellule de **vérification** qui vous dit
immédiatement si votre réponse est bonne.

**Partie 2 — les questions.** Une question, une cellule **vide** : à vous d'écrire le
code entier. Il n'y a pas de vérification automatique — on les corrige ensemble en
séance, et la correction est publiée après.

> ⚠️ Si une vérification de la partie 1 affiche `NameError`, c'est que la cellule
au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la,
puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/Intelligence-Artificielle-et-Data-Science/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]     ## le CA de chaque ligne
ventes["date"] = pd.to_datetime(ventes["date"])   ## du texte vers des dates

print(ventes.shape, clients.shape, produits.shape)

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Le meilleur client

> **Votre mission :**
> - Calculer le chiffre d'affaires par client.
> - Mettre l'identifiant du meilleur dans `meilleur_client` et son CA dans `ca_meilleur` (arrondi à 2 décimales).

In [ ]:
ca_client = ventes.groupby("____")["ca"].____()

meilleur_client = ca_client.idxmax()
ca_meilleur = round(ca_client.____(), 2)

print(meilleur_client, ":", ca_meilleur, "euros")

In [ ]:
verifier("1a - meilleur client", meilleur_client == 14911,
         "groupby sur client_id puis sum() sur la colonne ca")
verifier("1b - son chiffre d'affaires", ca_meilleur == 143825.06,
         "idxmax() renvoie l'identifiant, max() renvoie le montant")

### Exercice 2 — Les deux jointures, et le classement des marchés

> **Votre mission :**
> - Joindre `ventes` et `clients` sur `client_id` → `vc`, puis `vc` et `produits` sur `prod_id` → `complet`.
> - **Vérifier après chaque jointure** que le nombre de lignes n'a pas changé.
> - Puis calculer le CA par pays, trié du plus grand au plus petit → `ca_pays`, et mettre celui de la France dans `ca_france` (arrondi à 2 décimales).

In [ ]:
vc = ventes.____(clients, on="____")
complet = vc.merge(produits, on="____")
print(len(ventes), "->", len(vc), "->", len(complet))

ca_pays = vc.groupby("pays")["ca"].sum().sort_values(ascending=____)
ca_france = round(ca_pays["____"], 2)

print(ca_pays.head(3).round(2))
print("France :", ca_france)

In [ ]:
verifier("2a - jointure clients sans perte", len(vc) == 45123,
         "un nombre different signale une cle de jointure non unique")
verifier("2b - jointure produits sans perte", len(complet) == 45123,
         "la cle commune entre vc et produits est prod_id")
verifier("2c - CA de la France", ca_france == 133984.8,
         "groupby('pays') puis sum() sur ca, et ca_pays['France']")

### Exercice 3 — Le panier moyen par pays

> **Votre mission :**
> - Pour chaque pays : le CA total (`ca`) et le nombre de **commandes distinctes** (`nb_cmd`).
> - Ajouter une colonne `panier` = CA ÷ nombre de commandes, arrondie à 2 décimales.
> - Mettre le panier moyen irlandais dans `panier_irl`.

In [ ]:
parpays = vc.groupby("pays").agg(
    ca=("ca", "sum"),
    nb_cmd=("cmd_id", "____"),
)
parpays["panier"] = (parpays["ca"] / parpays["____"]).round(2)

panier_irl = parpays.loc["Irlande", "panier"]
print(panier_irl)

In [ ]:
verifier("3 - panier moyen irlandais", panier_irl == 1020.33,
         "avec count au lieu de nunique le panier serait ridiculement bas")

### Exercice 4 — Le tableau croisé

> **Votre mission :**
> - Croiser `pays` (en lignes) et `segment` (en colonnes), avec la somme du `ca` → `tableau`.
> - Mettre le CA des clients « premium » français dans `fr_premium` (arrondi à 0 décimale).

In [ ]:
tableau = vc.pivot_table(values="ca", index="____", columns="____", aggfunc="sum")

fr_premium = round(tableau.loc["France", "premium"], 0)
print(fr_premium)

In [ ]:
verifier("4 - premium francais", fr_premium == 114433.0,
         "index=pays (lignes), columns=segment (colonnes), aggfunc='sum'")

### Exercice 5 — La courbe, avec titre et unité

> **Votre mission :**
> - Tracer le **nombre de commandes distinctes** par mois sous forme de courbe.
> - Titre et unité obligatoires : un graphique sans légende n'est pas un graphique, c'est un dessin.
> - Incliner les étiquettes à 45° et appeler `tight_layout()` : sur un petit écran, sans ça les dates se chevauchent ou sont coupées.
> - Mettre le mois qui compte le plus de commandes dans `mois_cmd`, et le nombre de mois du fichier dans `nb_mois`.

In [ ]:
nb_cmd = complet.groupby(complet["date"].dt.to_period("M"))["cmd_id"].____()
nb_cmd.index = nb_cmd.index.astype(str)

nb_cmd.plot(kind="____", marker="o", figsize=(7, 4))
plt.title("Nombre de commandes par mois")
plt.ylabel("____")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

mois_cmd = nb_cmd.idxmax()
nb_mois = len(nb_cmd)
print(mois_cmd, "|", nb_mois, "mois")

In [ ]:
verifier("5a - mois record en commandes", mois_cmd == "2011-11",
         "nunique() compte les commandes distinctes, count() compterait les lignes")
verifier("5b - nombre de mois", nb_mois == 13,
         "decembre 2010 et decembre 2011 comptent tous les deux")

### Exercice 6 — Les barres, et le bon graphique pour la bonne question

> **Votre mission :**
> - Calculer le CA par **jour de la semaine** dans `ca_jour`, trié du plus petit au plus grand, et le tracer en barres horizontales avec titre et unité. Mettre le jour le plus fort dans `jour_top`.
> - Puis compléter `reponses` avec les quatre types de graphique, **dans l'ordre des questions** :
> - 1. Comment le chiffre d'affaires évolue-t-il dans le temps ? · 2. Quel pays est le plus gros marché ? · 3. Comment les prix sont-ils répartis ? · 4. Les grosses quantités vont-elles avec les prix bas ?

In [ ]:
ca_jour = complet.groupby(complet["date"].dt.day_name())["ca"].sum().____()

ca_jour.plot(kind="____", figsize=(7, 4))
plt.title("Chiffre d'affaires par jour de la semaine")
plt.xlabel("CA (euros)")
plt.tight_layout()
plt.show()

jour_top = ca_jour.index[-1]
reponses = ["____", "____", "____", "____"]
print(jour_top, "|", reponses)

In [ ]:
verifier("6a - jour le plus fort", jour_top == "Thursday",
         "sort_values() trie ; apres un tri croissant le plus grand est en position -1")
verifier("6b - le bon graphique pour la bonne question",
         reponses == ["line", "barh", "hist", "scatter"],
         "une evolution, un classement, une repartition, une relation")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 7 — La saisonnalité, et le piège de décembre

> **Votre mission :**
> - **Le contexte.** Votre direction prépare le budget de l'an prochain : **où est le chiffre d'affaires, et où sont les risques ?** Les six questions qui suivent répondent à cette question, une pièce à la fois.
> - Calculer le CA par mois dans `ca_mois` (index = le mois en texte, ex. `"2011-10"`), le tracer en **courbe** avec titre et libellé d'axe, et mettre le meilleur mois dans `mois_top`.
> - La courbe montre une chute en décembre. **Avant de conclure**, comptez les jours de décembre présents dans les données → `jours_dec`.
> - Puis répondez : la chute est-elle réelle ? Mettez `True` ou `False` dans `chute_reelle`, et expliquez en commentaire ce que dit vraiment le pic d'octobre.

### Question 8 — Les marchés

> **Votre mission :**
> - CA par pays, top 8, en **barres horizontales** triées, avec titre et unité.
> - Mettre le deuxième marché dans `marche_2`.

### Question 9 — La concentration client

> **Votre mission :**
> - Calculer le CA par client, puis la part des **10 premiers** dans le CA total → `part_top10` (en %, arrondi à 1 décimale).
> - Compter les clients irlandais → `nb_irl`.
> - Ces deux chiffres ne valent qu'ensemble. Que disent-ils ?

### Question 10 — Le top produits, et ce qu'il révèle

> **Votre mission :**
> - Afficher les 5 produits qui génèrent le plus de CA → `top_prod`.
> - **Regardez les noms attentivement.** Deux d'entre eux ne sont pas des produits.
> - Mettre leurs deux libellés dans la liste `faux_produits`.

### Question 11 — Le classement corrigé

> **Votre mission :**
> - Refaire le top 5 en excluant `Postage` et `Manual` → `top_reel`.
> - Calculer la part de ces deux lignes dans le CA total → `part_faux` (en %, arrondi à 1 décimale).
> - *Nouveau :* `query("libelle not in @faux_produits")` — le `@` va chercher une variable Python, `not in` inverse l'appartenance.

### Question 12 — Ce que vous en concluez

> **Votre mission :**
> - La cellule ci-dessous rassemble les chiffres des questions 7 à 11. Exécutez-la, puis rédigez votre conclusion dans une **cellule de texte** que vous ajouterez en dessous.
> - **Trois recommandations, chacune appuyée sur un chiffre que vous avez calculé.**
> - Un constat n'est pas une recommandation : « l'Irlande fait 22,7 % du CA » est un constat ; « il faut sécuriser ces deux contrats » est une recommandation.

In [ ]:
# Code fourni : executez-le tel quel. Le travail est la cellule de
# TEXTE que vous ajouterez en dessous (bouton "+ Texte").
print("meilleur mois        :", mois_top)
print("2e marche            :", marche_2, "avec", nb_irl, "clients")
print("part des 10 premiers :", part_top10, "%")
print("faux produits        :", part_faux, "% du CA")